# Segmentación de empresas eléctricas con clustering aplicado al VAD

- Clustering aplicado a indicadores de costo de empresas de distribución eléctrica, inspirado en el criterio regulatorio chileno de Áreas Típicas de Distribución (ATD) para el cálculo del Valor Agregado de Distribución (VAD).
- Incluye, como insumo complementario, la exploración de un perfil de carga de alimentador, usado para estimar costos teóricos de infraestructura.
- Artículo completo, con el diagrama del pipeline y las decisiones de ingeniería: https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/segmentacion-empresas-electricas-clustering-vad/
- **Nota:** los datos usados aquí son sintéticos, generados para fines demostrativos y académicos. Replican la estructura de datos reales de empresa, nunca sus valores ni ningún indicador que permita identificarla.


## Diagrama del pipeline

- Indicadores por empresa (VADT, CxK) → normalización → clustering (K-Means) → comparación contra el criterio regulatorio real.
- En paralelo: perfil de carga de un alimentador → exploración estadística → insumo para el costo teórico de infraestructura.
- Ambos análisis convergen en el mismo objetivo: un insumo técnico para el proceso tarifario, nunca la tarifa final.
- Diagrama editable disponible en el artículo de la plataforma (liga arriba).


## Carga de datos

- `91_DataIn_sintetico.csv`: indicadores sintéticos por empresa (VADT, CxK, número de clientes, km de red), con la misma estructura que el dato real de origen.
- `Perfil_alimentador_sintetico.csv`: perfil de carga sintético de una semana, cada 15 minutos, con la misma estructura que el perfil real de un alimentador.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

datos_empresas = pd.read_csv("91_DataIn_sintetico.csv")
datos_empresas.head()


## Explicación de datos

- `VADT`: valor agregado de distribución total, uno de los indicadores de costo por empresa.
- `CxK`: costo por cliente, un indicador de eficiencia relativa entre empresas.
- El resto de columnas (número de clientes, km de red) no se usan directamente en el clustering, pero ayudan a interpretar por qué una empresa cae en un grupo u otro.


In [ ]:
print(f"Empresas en el dataset: {len(datos_empresas)}")
datos_empresas[["VADT", "CxK"]].describe()


## Análisis de datos / EDA

- Antes de agrupar, conviene ver la dispersión de VADT y CxK, y confirmar que no hay valores atípicos extremos que distorsionen el clustering.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(datos_empresas["CxK"], datos_empresas["VADT"], alpha=0.7)
plt.xlabel("CxK (costo por cliente)")
plt.ylabel("VADT (valor agregado de distribución total)")
plt.title("Dispersión de empresas antes de normalizar")
plt.grid(True)
plt.show()


## Modelado

- Se normalizan VADT y CxK dividiendo entre su valor máximo, para que ninguna variable domine la distancia del clustering solo por su escala.
- Se aplica K-Means con 6 clústeres, un número exploratorio razonable para dos variables, no una réplica de las 12 Áreas Típicas de Distribución oficiales.


In [ ]:
VADTx1 = datos_empresas["VADT"] / datos_empresas["VADT"].max()
CxKx1 = datos_empresas["CxK"] / datos_empresas["CxK"].max()

datos_normalizados = pd.DataFrame({"CxKx1": CxKx1, "VADTx1": VADTx1})
datos_normalizados.index = datos_empresas["Empresa"]

N_CLUSTERS = 6
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=500, random_state=42)
kmeans.fit(datos_normalizados)

datos_empresas["Cluster"] = kmeans.predict(datos_normalizados)
centroides = kmeans.cluster_centers_
datos_empresas[["Empresa", "VADT", "CxK", "Cluster"]].head(10)


## Evaluación

- No hay una etiqueta "correcta" por empresa para validar el clustering de forma supervisada, así que la evaluación es principalmente visual e interpretativa.
- Se grafican los clústeres y sus centroides, y se revisa si el número de grupos resultante tiene sentido frente al criterio regulatorio de 12 Áreas Típicas de Distribución reales.


In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(centroides[:, 0], centroides[:, 1], alpha=0.3, s=500, c="black", label="Centroides")
plt.scatter(CxKx1, VADTx1, c=datos_empresas["Cluster"], s=80, cmap="rainbow")
plt.xlabel("CxK normalizado")
plt.ylabel("VADT normalizado")
plt.title(f"Segmentación de empresas en {N_CLUSTERS} clústeres")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Exploración del perfil de carga del alimentador (insumo complementario para costos teóricos)
perfil = pd.read_csv("Perfil_alimentador_sintetico.csv")
perfil.info()
perfil.describe()


In [ ]:
perfil[["kWD", "kVarD", "kWR", "kVarR"]].hist(bins=30, figsize=(12, 8))
plt.tight_layout()
plt.show()


## Hallazgos principales

- El clustering con K-Means, sobre solo dos variables normalizadas, agrupa razonablemente bien a las empresas por estructura de costos, el mismo principio detrás de las Áreas Típicas de Distribución del regulador chileno.
- El número de clústeres del ejercicio exploratorio (6) no coincide con el número real de Áreas Típicas oficiales (12), una diferencia esperada al trabajar con un espacio de variables mucho más simple que el estudio regulatorio completo.
- El perfil de carga del alimentador es un insumo distinto pero complementario, alimenta el cálculo del costo teórico de infraestructura, el otro componente central del VAD.
- Ni los indicadores de empresa ni el perfil de carga son datos abiertos declarados. Cuando la fuente es una empresa real, la autorización y la anonimización, incluyendo el uso de datos sintéticos como los de este notebook, son parte legítima del proceso, no un atajo.
